# 1. Setup and run given student.py

Should give a NotImplementedError

In [1]:
!curl -LsSf https://astral.sh/uv/install.sh | sh

downloading uv 0.10.12 x86_64-unknown-linux-gnu
no checksums to verify
installing to /usr/local/bin
  uv
  uvx
everything's installed!


In [2]:
!git clone https://github.com/baahl-nyu/ECE-9413.git

Cloning into 'ECE-9413'...
remote: Enumerating objects: 129, done.
remote: Counting objects: 100% (13/13), done.
remote: Compressing objects: 100% (10/10), done.
remote: Total 129 (delta 3), reused 6 (delta 3), pack-reused 116 (from 2)
Receiving objects: 100% (129/129), 414.16 MiB | 15.34 MiB/s, done.
Resolving deltas: 100% (15/15), done.
Updating files: 100% (101/101), done.


In [3]:
%cd ECE-9413/assignment1/


/content/ECE-9413/assignment1


In [4]:
!bash scripts/setup.sh

Installing with --extra cuda13
Using CPython 3.12.12 interpreter at: /usr/bin/python3
Creating virtual environment at: .venv
Resolved 56 packages in 1.21s
Prepared 33 packages in 24.92s
Installed 33 packages in 59ms
 + iniconfig==2.3.0
 + jax==0.9.2
 + jax-cuda13-pjrt==0.9.2
 + jax-cuda13-plugin==0.9.2
 + jaxlib==0.9.2
 + markdown-it-py==4.0.0
 + mdurl==0.1.2
 + ml-dtypes==0.5.4
 + mpmath==1.3.0
 + numpy==2.4.3
 + nvidia-cublas==13.3.0.5
 + nvidia-cuda-cccl==13.2.27
 + nvidia-cuda-crt==13.2.51
 + nvidia-cuda-cupti==13.2.23
 + nvidia-cuda-nvcc==13.2.51
 + nvidia-cuda-nvrtc==13.2.51
 + nvidia-cuda-runtime==13.2.51
 + nvidia-cudnn-cu13==9.20.0.48
 + nvidia-cufft==12.2.0.37
 + nvidia-cusolver==12.1.0.51
 + nvidia-cusparse==12.7.9.17
 + nvidia-nccl-cu13==2.29.7
 + nvidia-nvjitlink==13.2.51
 + nvidia-nvshmem-cu13==3.5.21
 + nvidia-nvvm==13.2.51
 + opt-einsum==3.4.0
 + packaging==26.0
 + pluggy==1.6.0
 + pygments==2.19.2
 + pytest==9.0.2
 + rich==14.3.3
 + scipy==1.17.1
 + sympy==1.14.0


In [5]:
%%writefile student.py

"""
Negacyclic Number Theoretic Transform (NTT) implementation.

The negacyclic NTT computes polynomial evaluation at odd powers of a primitive
root. Given coefficients x[0], x[1], ..., x[N-1], the output is:

    y[k] = Σ_{n=0}^{N-1} x[n] · ψ^{(2k+1)·n}  (mod q)

where ψ is a primitive 2N-th root of unity (ψ^N ≡ -1 mod q).

This is equivalent to a cyclic NTT on "twisted" input, where each coefficient
x[n] is first multiplied by ψ^n.
"""

import jax.numpy as jnp


# -----------------------------------------------------------------------------
# Modular Arithmetic
# -----------------------------------------------------------------------------

def mod_add(a, b, q):
    """Return (a + b) mod q, elementwise."""
    raise NotImplementedError


def mod_sub(a, b, q):
    """Return (a - b) mod q, elementwise."""
    raise NotImplementedError


def mod_mul(a, b, q):
    """Return (a * b) mod q, elementwise."""
    raise NotImplementedError


# -----------------------------------------------------------------------------
# Core NTT
# -----------------------------------------------------------------------------


def ntt(x, *, q, psi_powers, twiddles):
    """
    Compute the forward negacyclic NTT.

    Args:
        x: Input coefficients, shape (batch, N), values in [0, q)
        q: Prime modulus satisfying (q - 1) % 2N == 0
        psi_powers: Precomputed ψ^n table
        twiddles: Precomputed twiddle table

    Returns:
        jnp.ndarray: NTT output, same shape as input
    """
    raise NotImplementedError


def prepare_tables(*, q, psi_powers, twiddles):
    """
    Optional one-time table preparation.

    Override this if you want faster modular multiplication than JAX's "%".
    For example, you can convert the provided tables into Montgomery form
    (or any other domain) once here, then run `ntt` using your mod_mul.
    This function runs before timing, so its cost is not counted as latency.
    Must return (psi_powers, twiddles) in the form expected by `ntt`.
    """
    return psi_powers, twiddles



Overwriting student.py


In [6]:
!uv run pytest

FFFFF                                                                    [100%]
=================================== FAILURES ===================================
____________________________ test_matches_reference ____________________________

batch = 4
ntt_params = (1024, 2147473409, 383167813, Array([         1,  383167813, 2094155704, ...,  101321468, 1509222076,
       106363104...ype=uint32), Array([         1,          1,          1, ..., 1850991065,  234004256,
       1509222076], dtype=uint32))

    def test_matches_reference(batch, ntt_params):
        """NTT output matches SymPy reference across sizes and batch shapes."""
        N, q, psi, psi_powers, twiddles = ntt_params
        prepare = getattr(student, "prepare_tables", None)
        if prepare is not None:
            psi_powers, twiddles = prepare(
                q=q, psi_powers=psi_powers, twiddles=twiddles
            )
    
        rng = np.random.default_rng(DEFAULT_SEED)
        x = random_input(rng, q, shape=(ba

# 2. Brendon's version


In [7]:
%%writefile student.py

"""
Negacyclic Number Theoretic Transform (NTT) implementation.

The negacyclic NTT computes polynomial evaluation at odd powers of a primitive
root. Given coefficients x[0], x[1], ..., x[N-1], the output is:

    y[k] = Σ_{n=0}^{N-1} x[n] · ψ^{(2k+1)·n}  (mod q)

where ψ is a primitive 2N-th root of unity (ψ^N ≡ -1 mod q).

This is equivalent to a cyclic NTT on "twisted" input, where each coefficient
x[n] is first multiplied by ψ^n.
"""

import jax.numpy as jnp
import jax


# -----------------------------------------------------------------------------
# Modular Arithmetic
# -----------------------------------------------------------------------------

def mod_add(a, b, q):
    """Return (a + b) mod q, elementwise."""
    return (a + b) % q


def mod_sub(a, b, q):
    """Return (a - b) mod q, elementwise."""
    return (a - b) % q


def mod_mul(a, b, q):
    """Return (a * b) mod q, elementwise."""
    return (a * b) % q


# -----------------------------------------------------------------------------
# Core NTT
# -----------------------------------------------------------------------------


def ntt(x, *, q, psi_powers, twiddles):
    """
    Compute the forward negacyclic NTT.

    Args:
        x: Input coefficients, shape (batch, N), values in [0, q)
        q: Prime modulus satisfying (q - 1) % 2N == 0
        psi_powers: Precomputed ψ^n table
        twiddles: Precomputed twiddle table

    Returns:
        jnp.ndarray: NTT output, same shape as input
    """

    batches, n = x.shape

    # 1. Corrected Matrix Formula
    def my_formula(i, j):
        # Cast to int64 to prevent any overflow during index calculation
        i = i.astype(jnp.int64)
        j = j.astype(jnp.int64)

        # The true exponent: (2k + 1) * n
        power_idx = ((2 * i + 1) * j) % (2 * n)

        # Handle the psi^N = -1 (mod q) property
        val = jnp.where(
            power_idx < n,
            psi_powers[power_idx],
            q - psi_powers[power_idx - n] # -x mod q is q - x
        )
        return val

    matrix = jnp.fromfunction(my_formula, (n, n), dtype=jnp.uint32)

    # 2. Safe Modular Dot Product
    def mod_dot_product(vec, row):
        # Upcast to uint64 BEFORE multiplying so large numbers don't overflow
        vec64 = vec.astype(jnp.uint64)
        row64 = row.astype(jnp.uint64)

        products = (vec64 * row64) % q

        # Sum, mod q, and explicitly cast back to uint32 to pass the test
        return (jnp.sum(products) % q).astype(jnp.uint32)

    # 3. Apply via vmap
    dot_vmap = jax.vmap(mod_dot_product, in_axes=(None, 0))
    batch_vmap = jax.vmap(dot_vmap, in_axes=(0, None))

    final_output = batch_vmap(x, matrix)

    return final_output





def prepare_tables(*, q, psi_powers, twiddles):
    """
    Optional one-time table preparation.

    Override this if you want faster modular multiplication than JAX's "%".
    For example, you can convert the provided tables into Montgomery form
    (or any other domain) once here, then run `ntt` using your mod_mul.
    This function runs before timing, so its cost is not counted as latency.
    Must return (psi_powers, twiddles) in the form expected by `ntt`.
    """
    return psi_powers, twiddles

Overwriting student.py


In [8]:
!uv run pytest

.....                                                                    [100%]
5 passed in 5.43s


# 3. Naive version

In [9]:
%%writefile student.py

import jax.numpy as jnp


def mod_add(a, b, q):
    """Return (a + b) mod q, elementwise."""
    a64 = jnp.asarray(a, dtype=jnp.uint64)
    b64 = jnp.asarray(b, dtype=jnp.uint64)
    q64 = jnp.asarray(q, dtype=jnp.uint64)
    return jnp.asarray((a64 + b64) % q64, dtype=jnp.uint32)


def mod_sub(a, b, q):
    """Return (a - b) mod q, elementwise."""
    a64 = jnp.asarray(a, dtype=jnp.uint64)
    b64 = jnp.asarray(b, dtype=jnp.uint64)
    q64 = jnp.asarray(q, dtype=jnp.uint64)
    return jnp.asarray((a64 + q64 - b64) % q64, dtype=jnp.uint32)


def mod_mul(a, b, q):
    """Return (a * b) mod q, elementwise."""
    a64 = jnp.asarray(a, dtype=jnp.uint64)
    b64 = jnp.asarray(b, dtype=jnp.uint64)
    q64 = jnp.asarray(q, dtype=jnp.uint64)
    return jnp.asarray((a64 * b64) % q64, dtype=jnp.uint32)


def ntt(x, *, q, psi_powers, twiddles):
    """
    Compute the forward negacyclic NTT.

    Args:
        x: Input coefficients, shape (batch, N), values in [0, q)
        q: Prime modulus satisfying (q - 1) % 2N == 0
        psi_powers: Precomputed ψ^n table
        twiddles: Precomputed twiddle table

    Returns:
        jnp.ndarray: NTT output, same shape as input, dtype uint32
    """
    del twiddles  # unused in the naive implementation

    x64 = jnp.asarray(x, dtype=jnp.uint64)
    q64 = jnp.asarray(q, dtype=jnp.uint64)
    psi_tbl = jnp.asarray(psi_powers, dtype=jnp.uint64)

    B, N = x64.shape
    tbl_len = psi_tbl.shape[0]

    k = jnp.arange(N, dtype=jnp.uint64)[:, None]   # (N, 1)
    n = jnp.arange(N, dtype=jnp.uint64)[None, :]   # (1, N)
    e = (2 * k + 1) * n                            # (N, N)

    if tbl_len >= 2 * N:
        # Table already contains enough powers to index mod 2N directly.
        A = psi_tbl[e % jnp.uint64(tbl_len)]
    else:
        # Table is only length N. Use psi^(a + bN) = psi^a * (-1)^b.
        e_mod = e % jnp.uint64(tbl_len)
        wraps = (e // jnp.uint64(tbl_len)) & jnp.uint64(1)

        vals = psi_tbl[e_mod]
        neg_vals = (q64 - vals) % q64
        A = jnp.where(wraps == 0, vals, neg_vals)

    # y[b, k] = sum_n x[b, n] * A[k, n] mod q
    products = (x64[:, None, :] * A[None, :, :]) % q64
    y64 = jnp.sum(products, axis=-1) % q64

    return jnp.asarray(y64, dtype=jnp.uint32)


def prepare_tables(*, q, psi_powers, twiddles):
    """
    Optional one-time table preparation.

    Override this if you want faster modular multiplication than JAX's "%".
    For example, you can convert the provided tables into Montgomery form
    (or any other domain) once here, then run `ntt` using your mod_mul.
    This function runs before timing, so its cost is not counted as latency.
    Must return (psi_powers, twiddles) in the form expected by `ntt`.
    """
    return psi_powers, twiddles


Overwriting student.py


In [10]:
!uv run pytest

.....                                                                    [100%]
5 passed in 4.87s


# 4. FFT version

In [11]:
%%writefile student.py

import jax
import jax.numpy as jnp


def mod_add(a, b, q):
    """Return (a + b) mod q, elementwise."""
    a64 = jnp.asarray(a, dtype=jnp.uint64)
    b64 = jnp.asarray(b, dtype=jnp.uint64)
    q64 = jnp.asarray(q, dtype=jnp.uint64)
    return jnp.asarray((a64 + b64) % q64, dtype=jnp.uint32)


def mod_sub(a, b, q):
    """Return (a - b) mod q, elementwise."""
    a64 = jnp.asarray(a, dtype=jnp.uint64)
    b64 = jnp.asarray(b, dtype=jnp.uint64)
    q64 = jnp.asarray(q, dtype=jnp.uint64)
    return jnp.asarray((a64 + q64 - b64) % q64, dtype=jnp.uint32)


def mod_mul(a, b, q):
    """Return (a * b) mod q, elementwise."""
    a64 = jnp.asarray(a, dtype=jnp.uint64)
    b64 = jnp.asarray(b, dtype=jnp.uint64)
    q64 = jnp.asarray(q, dtype=jnp.uint64)
    return jnp.asarray((a64 * b64) % q64, dtype=jnp.uint32)


def _psi_pow_from_table(psi_powers, e, q):
    """
    Recover psi^e mod q from a length-N table psi_powers[n] = psi^n for n=0..N-1,
    using psi^N == -1 mod q.
    """
    psi_powers = jnp.asarray(psi_powers, dtype=jnp.uint64)
    q64 = jnp.asarray(q, dtype=jnp.uint64)
    N = psi_powers.shape[0]

    e = jnp.asarray(e, dtype=jnp.uint64) % jnp.uint64(2 * N)
    lo = e % jnp.uint64(N)
    hi = e // jnp.uint64(N)  # 0 or 1 after mod 2N

    vals = psi_powers[lo]
    neg_vals = (q64 - vals) % q64
    return jnp.where(hi == 0, vals, neg_vals)


def _bit_reverse_indices(N):
    logN = N.bit_length() - 1
    out = []
    for i in range(N):
        r = 0
        x = i
        for _ in range(logN):
            r = (r << 1) | (x & 1)
            x >>= 1
        out.append(r)
    return jnp.asarray(out, dtype=jnp.int32)


def prepare_tables(*, q, psi_powers, twiddles):
    """
    Precompute tables for radix-2 CT NTT.

    Returns:
        psi_powers unchanged
        twiddles repacked as:
          (bitrev, stage_twiddles)

    where:
      bitrev: permutation to bit-reverse the input
      stage_twiddles[s]: shape (2^s,), containing powers of omega needed
                         at stage s, with omega = psi^2
    """
    del twiddles

    psi_powers = jnp.asarray(psi_powers, dtype=jnp.uint32)
    N = int(psi_powers.shape[0])
    logN = N.bit_length() - 1

    bitrev = _bit_reverse_indices(N)

    # omega = psi^2, so omega^t = psi^(2t)
    t = jnp.arange(N, dtype=jnp.uint64)
    omega_powers = jnp.asarray(_psi_pow_from_table(psi_powers, 2 * t, q), dtype=jnp.uint32)

    stage_twiddles = []
    for s in range(logN):
        m = 1 << (s + 1)
        half = m >> 1
        step = N >> (s + 1)
        exponents = jnp.arange(half, dtype=jnp.int32) * step
        stage_twiddles.append(omega_powers[exponents])

    return psi_powers, (bitrev, tuple(stage_twiddles))


def ntt(x, *, q, psi_powers, twiddles):
    """
    Fast forward negacyclic NTT using:
      1) twist by psi^n
      2) bit-reverse input
      3) iterative radix-2 Cooley-Tukey butterflies with omega = psi^2
    """
    bitrev, stage_twiddles = twiddles

    x = jnp.asarray(x, dtype=jnp.uint32)
    psi_powers = jnp.asarray(psi_powers, dtype=jnp.uint32)
    q64 = jnp.asarray(q, dtype=jnp.uint64)

    B, N = x.shape

    # Negacyclic twist: a[n] = x[n] * psi^n
    a = jnp.asarray(
        (x.astype(jnp.uint64) * psi_powers[None, :].astype(jnp.uint64)) % q64,
        dtype=jnp.uint32,
    )

    # Bit-reverse input so DIT CT gives natural-order output
    a = a[:, bitrev]

    # Iterative radix-2 CT
    for s, w_stage in enumerate(stage_twiddles):
        m = 1 << (s + 1)
        half = m >> 1

        a3 = a.reshape(B, N // m, m)

        left = a3[:, :, :half].astype(jnp.uint64)
        right = a3[:, :, half:].astype(jnp.uint64)
        w = jnp.asarray(w_stage, dtype=jnp.uint32).astype(jnp.uint64)[None, None, :]

        t = (right * w) % q64
        top = (left + t) % q64
        bot = (left + q64 - t) % q64

        a = jnp.concatenate(
            [top.astype(jnp.uint32), bot.astype(jnp.uint32)],
            axis=2,
        ).reshape(B, N)

    return jnp.asarray(a, dtype=jnp.uint32)

Overwriting student.py


In [12]:
!uv run pytest

.....                                                                    [100%]
5 passed in 22.40s


# 5. Overhead removed


In [13]:
%%writefile student.py

import jax
import jax.numpy as jnp


def mod_add(a, b, q):
    """Return (a + b) mod q, elementwise."""
    a64 = jnp.asarray(a, dtype=jnp.uint64)
    b64 = jnp.asarray(b, dtype=jnp.uint64)
    q64 = jnp.asarray(q, dtype=jnp.uint64)
    return jnp.asarray((a64 + b64) % q64, dtype=jnp.uint32)


def mod_sub(a, b, q):
    """Return (a - b) mod q, elementwise."""
    a64 = jnp.asarray(a, dtype=jnp.uint64)
    b64 = jnp.asarray(b, dtype=jnp.uint64)
    q64 = jnp.asarray(q, dtype=jnp.uint64)
    return jnp.asarray((a64 + q64 - b64) % q64, dtype=jnp.uint32)


def mod_mul(a, b, q):
    """Return (a * b) mod q, elementwise."""
    a64 = jnp.asarray(a, dtype=jnp.uint64)
    b64 = jnp.asarray(b, dtype=jnp.uint64)
    q64 = jnp.asarray(q, dtype=jnp.uint64)
    return jnp.asarray((a64 * b64) % q64, dtype=jnp.uint32)


def _psi_pow_from_table(psi_powers, e, q):
    """
    Recover psi^e mod q from a length-N table psi_powers[n] = psi^n for n=0..N-1,
    using psi^N == -1 mod q.
    """
    psi_powers = jnp.asarray(psi_powers, dtype=jnp.uint64)
    q64 = jnp.asarray(q, dtype=jnp.uint64)
    N = psi_powers.shape[0]

    e = jnp.asarray(e, dtype=jnp.uint64) % jnp.uint64(2 * N)
    lo = e % jnp.uint64(N)
    hi = e // jnp.uint64(N)

    vals = psi_powers[lo]
    neg_vals = (q64 - vals) % q64
    return jnp.where(hi == 0, vals, neg_vals)


def _bit_reverse_indices(N):
    logN = N.bit_length() - 1
    out = []
    for i in range(N):
        r = 0
        x = i
        for _ in range(logN):
            r = (r << 1) | (x & 1)
            x >>= 1
        out.append(r)
    return jnp.asarray(out, dtype=jnp.int32)


def prepare_tables(*, q, psi_powers, twiddles):
    """
    Precompute everything needed by the CT butterfly implementation.

    Returns:
        psi_powers: twist table, kept as uint32
        twiddles: tuple(bitrev, stage_twiddles)

    where:
      bitrev: shape (N,), bit-reversal permutation
      stage_twiddles[s]: shape (2^s,), twiddle factors for CT stage s
    """
    del twiddles

    psi_powers = jnp.asarray(psi_powers, dtype=jnp.uint32)
    N = int(psi_powers.shape[0])
    logN = N.bit_length() - 1

    bitrev = _bit_reverse_indices(N)

    # omega = psi^2, and omega^t = psi^(2t)
    t = jnp.arange(N, dtype=jnp.uint64)
    omega_powers = jnp.asarray(
        _psi_pow_from_table(psi_powers, 2 * t, q),
        dtype=jnp.uint32,
    )

    stage_twiddles = []
    for s in range(logN):
        m = 1 << (s + 1)
        half = m >> 1
        step = N >> (s + 1)
        exponents = jnp.arange(half, dtype=jnp.int32) * step
        stage_twiddles.append(omega_powers[exponents])

    return psi_powers, tuple([bitrev] + stage_twiddles)


@jax.jit
def ntt(x, *, q, psi_powers, twiddles):
    """
    Fast forward negacyclic NTT using:
      1) twist by psi^n
      2) bit-reverse input
      3) iterative radix-2 Cooley-Tukey butterflies with omega = psi^2
    """
    x = jnp.asarray(x, dtype=jnp.uint32)
    psi_powers = jnp.asarray(psi_powers, dtype=jnp.uint32)
    q64 = jnp.asarray(q, dtype=jnp.uint64)

    bitrev = twiddles[0]
    stage_twiddles = twiddles[1:]

    B, N = x.shape

    # Negacyclic twist: a[n] = x[n] * psi^n
    a = jnp.asarray(
        (x.astype(jnp.uint64) * psi_powers[None, :].astype(jnp.uint64)) % q64,
        dtype=jnp.uint32,
    )

    # Bit-reverse input so CT gives natural-order output
    a = a[:, bitrev]

    for s, w_stage in enumerate(stage_twiddles):
        m = 1 << (s + 1)
        half = m >> 1

        a3 = a.reshape(B, N // m, m)

        left = a3[:, :, :half].astype(jnp.uint64)
        right = a3[:, :, half:].astype(jnp.uint64)
        w = jnp.asarray(w_stage, dtype=jnp.uint32).astype(jnp.uint64)[None, None, :]

        t = (right * w) % q64
        top = (left + t) % q64
        bot = (left + q64 - t) % q64

        a = jnp.concatenate(
            [top.astype(jnp.uint32), bot.astype(jnp.uint32)],
            axis=2,
        ).reshape(B, N)

    return jnp.asarray(a, dtype=jnp.uint32)

Overwriting student.py


In [14]:
!uv run pytest

.....                                                                    [100%]
5 passed in 8.89s


# 6. Concat removed

In [15]:
%%writefile student.py

import jax
import jax.numpy as jnp


def mod_add(a, b, q):
    """Return (a + b) mod q, elementwise."""
    a64 = jnp.asarray(a, dtype=jnp.uint64)
    b64 = jnp.asarray(b, dtype=jnp.uint64)
    q64 = jnp.asarray(q, dtype=jnp.uint64)
    return jnp.asarray((a64 + b64) % q64, dtype=jnp.uint32)


def mod_sub(a, b, q):
    """Return (a - b) mod q, elementwise."""
    a64 = jnp.asarray(a, dtype=jnp.uint64)
    b64 = jnp.asarray(b, dtype=jnp.uint64)
    q64 = jnp.asarray(q, dtype=jnp.uint64)
    return jnp.asarray((a64 + q64 - b64) % q64, dtype=jnp.uint32)


def mod_mul(a, b, q):
    """Return (a * b) mod q, elementwise."""
    a64 = jnp.asarray(a, dtype=jnp.uint64)
    b64 = jnp.asarray(b, dtype=jnp.uint64)
    q64 = jnp.asarray(q, dtype=jnp.uint64)
    return jnp.asarray((a64 * b64) % q64, dtype=jnp.uint32)


def _psi_pow_from_table(psi_powers, e, q):
    """
    Recover psi^e mod q from a length-N table psi_powers[n] = psi^n for n=0..N-1,
    using psi^N == -1 mod q.
    """
    psi_powers = jnp.asarray(psi_powers, dtype=jnp.uint64)
    q64 = jnp.asarray(q, dtype=jnp.uint64)
    N = psi_powers.shape[0]

    e = jnp.asarray(e, dtype=jnp.uint64) % jnp.uint64(2 * N)
    lo = e % jnp.uint64(N)
    hi = e // jnp.uint64(N)

    vals = psi_powers[lo]
    neg_vals = (q64 - vals) % q64
    return jnp.where(hi == 0, vals, neg_vals)


def _bit_reverse_indices(N):
    logN = N.bit_length() - 1
    out = []
    for i in range(N):
        r = 0
        x = i
        for _ in range(logN):
            r = (r << 1) | (x & 1)
            x >>= 1
        out.append(r)
    return jnp.asarray(out, dtype=jnp.int32)


def prepare_tables(*, q, psi_powers, twiddles):
    """
    Precompute everything needed by the CT butterfly implementation.

    Returns:
        psi_powers: twist table, kept as uint32
        twiddles: tuple(bitrev, stage_twiddles)

    where:
      bitrev: shape (N,), bit-reversal permutation
      stage_twiddles[s]: shape (2^s,), twiddle factors for CT stage s
    """
    del twiddles

    psi_powers = jnp.asarray(psi_powers, dtype=jnp.uint32)
    N = int(psi_powers.shape[0])
    logN = N.bit_length() - 1

    bitrev = _bit_reverse_indices(N)

    # omega = psi^2, and omega^t = psi^(2t)
    t = jnp.arange(N, dtype=jnp.uint64)
    omega_powers = jnp.asarray(
        _psi_pow_from_table(psi_powers, 2 * t, q),
        dtype=jnp.uint32,
    )

    stage_twiddles = []
    for s in range(logN):
        m = 1 << (s + 1)
        half = m >> 1
        step = N >> (s + 1)
        exponents = jnp.arange(half, dtype=jnp.int32) * step
        stage_twiddles.append(omega_powers[exponents])

    return psi_powers, tuple([bitrev] + stage_twiddles)


@jax.jit
def ntt(x, *, q, psi_powers, twiddles):
    """
    Fast forward negacyclic NTT using:
      1) twist by psi^n
      2) bit-reverse input
      3) iterative radix-2 Cooley-Tukey butterflies with omega = psi^2
    """
    x = jnp.asarray(x, dtype=jnp.uint32)
    psi_powers = jnp.asarray(psi_powers, dtype=jnp.uint32)
    q64 = jnp.asarray(q, dtype=jnp.uint64)

    bitrev = twiddles[0]
    stage_twiddles = twiddles[1:]

    B, N = x.shape

    # Negacyclic twist: a[n] = x[n] * psi^n
    a = jnp.asarray(
        (x.astype(jnp.uint64) * psi_powers[None, :].astype(jnp.uint64)) % q64,
        dtype=jnp.uint32,
    )

    # Bit-reverse input so CT gives natural-order output
    a = a[:, bitrev]

    for s, w_stage in enumerate(stage_twiddles):
        m = 1 << (s + 1)
        half = m >> 1

        a3 = a.reshape(B, N // m, m)

        left = a3[:, :, :half].astype(jnp.uint64)
        right = a3[:, :, half:].astype(jnp.uint64)
        w = jnp.asarray(w_stage, dtype=jnp.uint32).astype(jnp.uint64)[None, None, :]

        t = (right * w) % q64
        top = ((left + t) % q64).astype(jnp.uint32)
        bot = ((left + q64 - t) % q64).astype(jnp.uint32)

        out = jnp.empty_like(a3)
        out = out.at[:, :, :half].set(top)
        out = out.at[:, :, half:].set(bot)

        a = out.reshape(B, N)

    return a

Overwriting student.py


In [16]:
!uv run pytest

.....                                                                    [100%]
5 passed in 29.96s
